# ReAct: Build Reasoning and Acting AI Agents with LangGraph

This project builds an AI agent that doesn't just respond — it **reasons**, then **acts**.

Using the **ReAct** (Reasoning + Acting) framework on LangGraph, the agent thinks a problem through step by step, calls tools such as web search or a calculator when it needs outside information, observes the results, and adapts until it can answer. The graph cycles between an *agent* node that decides what to do and a *tools* node that executes, ending only when no further tool calls are needed.

The worked example answers questions that knowledge alone can't solve — like *"what's the weather in Zurich, and what should I wear?"* — which requires live data plus reasoning over it.

## Table of Contents

1. Setup and installation
2. Understanding tools in ReAct
3. The reasoning model and system prompt
4. Agent state
5. Manual ReAct execution (understanding the flow)
6. Automating ReAct with a graph
7. Running the complete agent
8. Extending the agent: calculator and news tools

## What this project covers

- Using the ReAct framework to solve multi-step problems with external tools
- Getting an agent to reason step by step, act, and adapt based on results
- Building custom tools and registering them with the model
- Wiring a cyclical agent/tools graph in LangGraph with conditional edges
- Streaming agent output to watch the reasoning unfold

----


## Setup & Installation


This project uses the following libraries:

- [`langgraph`](https://langchain-ai.github.io/langgraph/) — build stateful, multi-step AI applications as graphs.
- [`langchain`](https://www.langchain.com/) — tools and abstractions for working with language models.
- [`langchain-groq`](https://pypi.org/project/langchain-groq/) — Groq-hosted LLMs.
- [`langchain-community`](https://python.langchain.com/api_reference/community/index.html) — community integrations (Serper search).
- [`langchain-tavily`](https://pypi.org/project/langchain-tavily/) — Tavily search, used when a Tavily key is available.
- [`python-dotenv`](https://pypi.org/project/python-dotenv/) — load API keys from `.env`.

### Installing Required Libraries


In [ ]:
%%capture
!pip install -U langgraph langchain langchain-groq langchain-community python-dotenv
!pip install -U langchain-tavily   # optional: only needed if you use Tavily for search

### Understanding Tools in ReAct

Tools are what let the agent *act*. Each is a plain Python function wrapped in `@tool`; the docstring is what the model reads to decide when to call it, so it doubles as the tool's specification. The cell below loads the API keys and defines the web-search tool.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json

from dotenv import load_dotenv, find_dotenv
from langchain.tools import tool

# Load API keys from a .env file (this folder or any parent directory).
# Never hard-code keys in a notebook you intend to share.
load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY is not set. Add it to your .env file before running.")
if not (os.environ.get("TAVILY_API_KEY") or os.environ.get("SERPER_API_KEY")):
    raise RuntimeError("Set TAVILY_API_KEY or SERPER_API_KEY in your .env file.")

# Web search: Tavily is used when its key is present, otherwise Serper.
if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    _search = TavilySearch(max_results=3)
    SEARCH_PROVIDER = "tavily"

    def _run_search(query: str):
        hits = _search.invoke({"query": query})
        if isinstance(hits, dict):
            hits = hits.get("results", [])
        return [
            {"title": h.get("title", ""), "link": h.get("url", ""),
             "snippet": (h.get("content", "") or "")[:300]}
            for h in hits[:3]
        ]
else:
    from langchain_community.utilities import GoogleSerperAPIWrapper

    _search = GoogleSerperAPIWrapper(k=3)
    SEARCH_PROVIDER = "serper"

    def _run_search(query: str):
        hits = _search.results(query).get("organic", [])
        return [
            {"title": h.get("title", ""), "link": h.get("link", ""),
             "snippet": h.get("snippet", "")}
            for h in hits[:3]
        ]


@tool
def search_tool(query: str):
    """
    Search the web for current information.

    :param query: The search query string
    :return: A list of {title, link, snippet} results
    """
    return _run_search(query)


print(f"search provider: {SEARCH_PROVIDER}")

### Why web search matters

- Retrieves information in real time
- Works around the model's knowledge cutoff
- Returns structured data the agent can reason over

### Testing the search tool

In [ ]:
search_tool.invoke({"query": "What's the weather like in Tokyo today?"})

This test demonstrates how the agent can access current information that wasn't available during training.

#### 2. Clothing Recommendation Tool


In [ ]:
@tool
def recommend_clothing(weather: str) -> str:
    """
    Returns a clothing recommendation based on the provided weather description.

    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.

    :param weather: A brief description of the weather (e.g., "Overcast, 64.9°F")
    :return: A string with clothing recommendations suitable for the weather
    """
    weather = weather.lower()
    if "snow" in weather or "freezing" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."

**Why this Tool Matters:**
- Demonstrates domain-specific reasoning
- Shows how tools can process and interpret data from other tools
- Illustrates the composability of ReAct systems

#### Creating the Tool Registry


In [ ]:
# The tools the agent can call at this stage.
tools = [search_tool, recommend_clothing]

tools_by_name = {tool.name: tool for tool in tools}
print("registered tools:", list(tools_by_name))

This registry allows the agent to dynamically select and invoke the appropriate tool based on the task at hand.

## Setting Up the Language Model

### Initializing the AI Model


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# The reasoning engine driving the ReAct loop.
#
# Model choice matters here: the agent has to emit well-formed tool calls on every
# turn, and hosted models vary in how reliably they do that. This one runs the whole
# notebook cleanly. "openai/gpt-oss-120b" reasons better but has a tighter rate
# limit; "llama-3.3-70b-versatile" tends to write the tool call out as text on
# multi-step queries, which the API rejects with tool_use_failed.
model = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

The Groq-hosted Llama model is the reasoning engine. It decides which tool to call, interprets what comes back, and composes the final answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful AI assistant that solves problems by reasoning and using tools.

How to work:
1. Decide what information you need to answer the question.
2. If you need current data or a capability a tool provides, call that tool.
   Use the tool-calling interface directly - never describe the call in prose and
   never write out function tags yourself.
3. Once a tool returns, reason over its results and call another tool if needed.
4. When you have enough information, give a clear final answer and explain how you
   reached it.
"""),
    MessagesPlaceholder(variable_name="scratch_pad")
])

**The System Prompt's Role:**
- Defines the agent's behavior and personality
- Establishes the reasoning pattern (think → act → observe)
- Encourages transparency in the decision-making process

### Binding Tools to the Model


In [ ]:
model_react=chat_prompt|model.bind_tools(tools)

This creates a model that can:
- Understand when to use tools
- Generate properly formatted tool calls
- Process tool results in context

## Understanding Agent State

### What is Agent State?

In ReAct, state management is crucial, as the agent must maintain context across multiple reasoning and acting steps.


In [ ]:
from typing import (Annotated,Sequence,TypedDict)
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """The state of the agent."""

    # add_messages is a reducer
    # See https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers
    messages: Annotated[Sequence[BaseMessage], add_messages]

**Key Concepts:**
- **State**: Contains the conversation history and context.
- **Reducer**: `add_messages` automatically handles adding new messages to the conversation.
- **Type Safety**: TypedDict ensures our state structure is well-defined.

### Demonstrating State Management


In [ ]:
# Example conversation flow:
state: AgentState = {"messages": []}

# append a message using the reducer properly
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

# add another message (e.g. a question)
state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)

This demonstrates how the state accumulates context over the conversation.


## Manual ReAct Execution (Understanding the Flow)

Before building the automated graph, let's manually step through a ReAct cycle to understand what happens:

### Step 1: Initial Query Processing


In [ ]:
dummy_state: AgentState = {
    "messages": [HumanMessage( "What's the weather like in Zurich, and what should I wear based on the temperature?")]}

response = model_react.invoke({"scratch_pad":dummy_state["messages"]})

dummy_state["messages"]=add_messages(dummy_state["messages"],[response])

**What Happens Here:**
1. The user asks a complex question requiring current data.
2. The model analyzes the query and realizes it needs to search for weather information.
3. The model generates a tool call for the search.


### Step 2: Tool Execution


In [ ]:
tool_call = response.tool_calls[-1]
print("Tool call:", tool_call)

tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])

# search_tool returns a list of result dicts; the other tools return a plain string,
# so print a preview that works either way.
if isinstance(tool_result, list) and tool_result:
    first = tool_result[0]
    print("Tool result preview:", first.get("title", first) if isinstance(first, dict) else first)
else:
    print("Tool result preview:", tool_result)

tool_message = ToolMessage(
    content=json.dumps(tool_result),
    name=tool_call["name"],
    tool_call_id=tool_call["id"]
)
dummy_state["messages"] = add_messages(dummy_state["messages"], [tool_message])

**What Happens Here:**
1. Extract the tool call from the model's response.
2. Execute the tool using the specified arguments.
3. Create a ToolMessage containing the results.
4. Add the tool result to the conversation state.


### Step 3: Processing Results and Next Action


In [ ]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state['messages'] = add_messages(dummy_state['messages'], [response])

# check if the model wants to use another tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=json.dumps(tool_result),
        name=tool_call["name"],
        tool_call_id=tool_call["id"]
    )
    dummy_state['messages'] = add_messages(dummy_state['messages'], [tool_message])

**What Happens Here:**
1. The model processes the search results.
2. It realizes it needs to use the clothing recommendation tool.
3. It extracts weather information and calls the clothing tool.
4. It receives clothing recommendations based on the weather data.


### Step 4: Final Response Generation


In [ ]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
print("Final response generated:", response.content is not None)
print("More tools needed:", bool(response.tool_calls))

**What Happens Here:**
1. The model has all necessary information.
2. It synthesizes weather data and clothing recommendations.
3. It generates a comprehensive response to the user.
4. No more tool calls needed—the reasoning cycle is complete.


## Automating ReAct with Graphs

### Why Use Graphs?

Manual ReAct execution is educational but impractical for real applications. LangGraph automates this process with a state machine that handles the reasoning loop automatically.

### Building the Core Functions

#### Tool Execution Node


In [ ]:
def tool_node(state: AgentState):
    """Execute all tool calls from the last message in the state."""
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}

**Function Purpose:**
- Automatically execute all tool calls from the model
- Handle multiple simultaneous tool calls
- Return properly formatted tool messages


#### Model Invocation Node


In [ ]:
def call_model(state: AgentState):
    """Invoke the model with the current conversation state."""
    response = model_react.invoke({"scratch_pad": state["messages"]})
    return {"messages": [response]}

**Function Purpose:**
- Call the ReAct-enabled model
- Pass the full conversation context
- Return the model's response (which may include tool calls)

#### Decision Logic


In [ ]:
def should_continue(state: AgentState):
    """Determine whether to continue with tool use or end the conversation."""
    messages = state["messages"]
    last_message = messages[-1]
    # If there is no function call, then we finish
    if not last_message.tool_calls:
        return "end"
    # Otherwise if there is, we continue
    else:
        return "continue"

**Function Purpose:**
- Implement the control flow logic
- Decide whether the agent needs to use more tools
- Route the conversation to either tool execution or completion

### Constructing the State Graph


In [ ]:
from langgraph.graph import StateGraph, END

# Define a new graph
workflow = StateGraph(AgentState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Add edges between nodes
workflow.add_edge("tools", "agent")  # After tools, always go back to agent

# Add conditional logic
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",  # If tools needed, go to tools node
        "end": END,          # If done, end the conversation
    },
)

# Set entry point
workflow.set_entry_point("agent")

# Compile the graph
graph = workflow.compile()

**Graph Structure Explained:**
1. **Agent Node**: Where reasoning happens and tool calls are generated.
2. **Tools Node**: Where tool execution occurs.
3. **Conditional Edge**: Determines whether to continue or finish.
4. **Entry Point**: Conversation always starts with the agent reasoning.
### Visualizing the Graph


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

This visualization shows the flow: Agent → Decision → Tools → Agent → Decision → End


## Running the Complete ReAct Agent

### Final Execution


In [ ]:
def print_stream(stream):
    """Helper function for formatting the stream nicely."""
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [HumanMessage(content="What's the weather like in Zurich, and what should I wear based on the temperature?")]}

print_stream(graph.stream(inputs, stream_mode="values"))

**What You'll See:**
1. **Initial Reasoning**: Agent analyzes the query.
2. **Tool Call 1**: Searches for Zurich weather.
3. **Tool Result Processing**: Agent examines weather data.
4. **Tool Call 2**: Gets clothing recommendations.
5. **Final Synthesis**: Agent combines all information into a helpful response.


### The Complete ReAct Cycle

The final execution demonstrates the full ReAct pattern:

1. **Reasoning**: "I need current weather data for Zurich".
2. **Acting**: Calls search_tool("Zurich weather today").
3. **Observing**: Processes search results, extracts temperature.
4. **Reasoning**: "Now I need clothing recommendations for this temperature".
5. **Acting**: Calls recommend_clothing("temperature from search").
6. **Observing**: Gets clothing suggestions.
7. **Reasoning**: "I can now provide a complete answer".
8. **Final Response**: Synthesizes weather info and clothing recommendations.


## Key Takeaways


### What Makes ReAct Powerful

1. **Transparency**: You can see the agent's reasoning process.
2. **Adaptability**: The agent can handle unexpected results and change course.
3. **Extensibility**: It's easy to add new tools and capabilities.
4. **Reliability**: The structured approach reduces hallucination and improves accuracy


### Best Practices

1. **Tool Design**: Make tools focused and reliable.
2. **Error Handling**: Plan for tool failures and unexpected results.
3. **Context Management**: Keep state manageable and relevant.
4. **User Experience**: Provide clear feedback about what the agent is doing.

The ReAct framework represents a significant step toward more capable and trustworthy AI agents that can reason through complex problems and take meaningful actions in the real world.


## Extending the Agent

Two more tools that broaden what the agent can handle: a calculator for exact arithmetic (something LLMs are unreliable at) and a summarizer for condensing search results.

### A calculator tool

Language models are unreliable at arithmetic, so give the agent a real calculator. Rather than calling `eval()` — which would execute any Python it's handed — this parses the expression with `ast` and walks the tree, allowing only known-safe operators and functions.

In [ ]:
import ast
import math
import operator

# Only these node types and operators are allowed, so arbitrary code can't run.
_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

_ALLOWED_NAMES = {"pi": math.pi, "e": math.e, "tau": math.tau}

_ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "log": math.log, "log10": math.log10, "exp": math.exp, "abs": abs,
    "round": round, "floor": math.floor, "ceil": math.ceil,
    "min": min, "max": max, "pow": math.pow,
}


def _evaluate(node):
    """Recursively evaluate a parsed expression, rejecting anything unsafe."""
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError(f"unsupported constant: {node.value!r}")

    if isinstance(node, ast.BinOp):
        op = _ALLOWED_OPERATORS.get(type(node.op))
        if op is None:
            raise ValueError("unsupported operator")
        return op(_evaluate(node.left), _evaluate(node.right))

    if isinstance(node, ast.UnaryOp):
        op = _ALLOWED_OPERATORS.get(type(node.op))
        if op is None:
            raise ValueError("unsupported unary operator")
        return op(_evaluate(node.operand))

    if isinstance(node, ast.Name):
        if node.id in _ALLOWED_NAMES:
            return _ALLOWED_NAMES[node.id]
        raise ValueError(f"unknown name: {node.id}")

    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name) or node.func.id not in _ALLOWED_FUNCTIONS:
            raise ValueError("unsupported function call")
        args = [_evaluate(a) for a in node.args]
        return _ALLOWED_FUNCTIONS[node.func.id](*args)

    raise ValueError(f"unsupported expression element: {type(node).__name__}")


@tool
def calculator_tool(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.

    Supports arithmetic (+ - * / // % **), the constants pi/e/tau, and common
    functions such as sqrt, sin, cos, tan, log, exp, abs, round, floor, ceil.

    :param expression: A mathematical expression, e.g. "2 + 3 * 4" or "sqrt(16)"
    :return: The result of the calculation, or an error message
    """
    try:
        tree = ast.parse(expression, mode="eval")
        result = _evaluate(tree.body)
    except Exception as exc:
        return f"Could not evaluate {expression!r}: {exc}"

    # Present whole numbers without a trailing .0
    if isinstance(result, float) and result.is_integer():
        result = int(result)
    return str(result)


# Quick checks
for expr in ["2 + 3 * 4", "sqrt(16)", "sin(pi/2)", "250 * 0.15 + sqrt(144)", "__import__('os')"]:
    print(f"{expr:<28} -> {calculator_tool.invoke({'expression': expr})}")

### A news summary tool

This searches for recent coverage of a topic and condenses it into a readable digest — headline, source, and key point per article.

Note the signature: it takes a short **topic**, not article text. It is tempting to have the agent pass raw search results in, but stuffing a large blob into a tool-call argument makes the model emit malformed JSON and the request gets rejected. Keeping arguments small and doing the retrieval inside the tool is both more reliable and less token-hungry.

In [ ]:
def _format_articles(articles) -> str:
    """Turn a list of {title, link, snippet} results into a readable digest."""
    if not articles:
        return "No articles found."

    lines = [f"Found {len(articles)} article(s):", ""]
    for i, article in enumerate(articles, start=1):
        headline = article.get("title") or "Untitled"
        link = article.get("link", "")
        snippet = (article.get("snippet") or "").strip().replace("\n", " ")
        if len(snippet) > 220:
            snippet = snippet[:220].rsplit(" ", 1)[0] + "..."

        # The domain stands in for the publication name.
        source = link.split("/")[2].replace("www.", "") if "//" in link else ""

        lines.append(f"{i}. {headline}")
        if source:
            lines.append(f"   Source: {source}")
        if snippet:
            lines.append(f"   Key point: {snippet}")
        if link:
            lines.append(f"   Link: {link}")
        lines.append("")

    return "\n".join(lines).strip()


@tool
def news_summarizer_tool(topic: str) -> str:
    """
    Search for recent news on a topic and return a readable summary.

    Pass a short topic, not article text - this tool does its own search and
    summarizes what it finds.

    :param topic: The news topic, e.g. "artificial intelligence"
    :return: A formatted summary listing each article's headline, source, and key point
    """
    if not topic or not topic.strip():
        return "No topic provided."
    return _format_articles(_run_search(f"latest news {topic}"))


# Quick check
print(news_summarizer_tool.invoke({"topic": "artificial intelligence"}))

### Testing the new tools

Rebuild the graph so it picks up the expanded tool list, then try questions that exercise the new capabilities.

In [ ]:
# Register the new tools, then rebuild the model binding and graph so they are picked up.
tools = [search_tool, recommend_clothing, calculator_tool, news_summarizer_tool]
tools_by_name = {tool.name: tool for tool in tools}
print("registered tools:", list(tools_by_name))

model_react = chat_prompt | model.bind_tools(tools)

workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge("tools", "agent")
workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
workflow.set_entry_point("agent")
graph = workflow.compile()

inputs = {"messages": [HumanMessage(content="Calculate 15% of 250 plus the square root of 144")]}
print_stream(graph.stream(inputs, stream_mode="values"))

In [ ]:
inputs = {"messages": [HumanMessage(content="Find recent AI news and summarize the top 3 articles")]}
print_stream(graph.stream(inputs, stream_mode="values"))

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)